# WorldSplat-Prune — TUM2TWIN base model (via 3DGS-Lightning)

WorldSplat-Prune is **post-reconstruction**: it prunes an *already-trained* 3DGS
model. This notebook produces that base model for the TUM2TWIN **UAS campus**
scene, in the Inria/GRAPHDECO layout the pruning + eval tooling expects
(`point_cloud/iteration_N/point_cloud.ply` + `cameras.json` + `cfg_args`).

**Why 3DGS-Lightning.** The official TUM2TWIN *Image2GaussianSplats* tutorial
recommends [yzslab/gaussian-splatting-lightning](https://github.com/yzslab/gaussian-splatting-lightning)
over the Inria reference for this large-scale scene (the Inria rasterizer OOMs).
This notebook trains there, then **exports back to an Inria-compatible model dir**
so `prune_model.py` / `eval_worldsplat.py` / `experiments/` work unchanged.

### No downloads here
The dataset zips are produced by a **separate retrieval notebook**
(`Retrieve_TUM2TWIN.ipynb`) and already live on mounted Drive at:

```
$RAW_DIR/tum2twin/uas_campus/UAS_Images.zip   # Zenodo Images.zip, ~5.9 GB
$RAW_DIR/tum2twin/uas_campus/opf.zip          # Pix4D OPF, ~1.9 GB (Zenodo 15282970)
```

This notebook **copies those zips to the local runtime** and does all heavy
multi-file work (unzip, undistortion, downsampling, training) on local disk —
mounted-Drive I/O is slow, especially for the ~1,179-file image set. Only
finished artifacts (the prepared scene, the model dir) are written back.

### Pipeline
1. Mount Drive; configure paths + knobs.
2. Copy `UAS_Images.zip` + `opf.zip` from Drive to local disk; extract locally.
3. OPF → COLMAP (`opf2colmap`), then `colmap model_converter` + `image_undistorter`
   → local `images/` + `sparse/0` (PINHOLE).
4. Set up 3DGS-Lightning in its own env (Miniconda py3.9 + CUDA 11.8 + torch 2.0.1+cu118).
5. Train **on the local scene**.
6. Export to an Inria-compatible model dir; sync the prepared scene to Drive.
7. Sanity evaluation with this repo's evaluator.

### Caveats that shaped this notebook
* **No SfM / no re-run.** Poses come from Pix4D's OPF calibration. Training
  resolution is chosen at *load* time (`down_sample_factor`), so you can switch
  between full-res and downscaled training **without rebuilding anything**.
* **`opf.zip` is version-dependent.** It does *not* exist in Zenodo v1.0
  (`14548134`, the record the tutorial links to); it was added in v1.1.0 and is
  present in the latest **v1.2.0 (`15282970`)**. Make sure your retrieval notebook
  fetched `opf.zip` from that version.
* **Full-res eval is VRAM-bound.** The Inria `Camera` loader preloads *every*
  image, so evaluating all ~1,179 4K frames at native resolution OOMs. The sanity
  step below evaluates at `-r 8`.
* **Two environments.** OPF/COLMAP conversion and the sanity eval run in Colab's
  base Python; 3DGS-Lightning runs in its own py3.9/cu118 conda env.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Configuration

**Drive (persistent):** `HOME_DIR` / `RAW_DIR` (zips live here) / `DATASET_ROOT`.
**Local (fast):** `SCRATCH` — all multi-file work happens here.

`DRIVE_ZIP_DIR` already matches `Retrieve_TUM2TWIN.ipynb`
(`$RAW_DIR/tum2twin/uas_campus`). `SCENE_DIR` is the *final* prepared scene on
Drive (for downstream pruning); training reads the **local** copy in
`SCENE_LOCAL`.

In [ ]:
import os
from pathlib import Path

# --- Drive layout (matches Retrieve_TUM2TWIN / Prepare_Datasets) -----------
HOME_DIR     = "/content/drive/MyDrive/Magister/Thesis/Script"
RAW_DIR      = Path(HOME_DIR) / "raw"
DATASET_ROOT = Path(HOME_DIR) / "Dataset"

# --- local runtime (fast) --------------------------------------------------
SCRATCH = Path("/content/_t2t")

SCENE_NAME = "uas_campus"

# --- where the retrieval notebook stored the zips (on Drive) ---------------
# Retrieve_TUM2TWIN persists: <RAW_DIR>/tum2twin/<SCENE_NAME>/{UAS_Images.zip,opf.zip}
DRIVE_ZIP_DIR   = RAW_DIR / "tum2twin" / SCENE_NAME
IMAGES_ZIP_NAME = "UAS_Images.zip"
OPF_ZIP_NAME    = "opf.zip"

# Final prepared scene on Drive (images/ + sparse/0/) for downstream pruning.
SCENE_DIR = DATASET_ROOT / "tum2twin" / SCENE_NAME

# Local working scene: what we actually undistort / downsample / train on.
SCENE_LOCAL = SCRATCH / "scene"

# Inria-format base model (small; keep on Drive).
MODEL_DIR = Path(HOME_DIR) / "output" / "tum2twin_uas_campus"

# --- 3DGS-Lightning --------------------------------------------------------
GSL_DIR   = Path("/content/gaussian-splatting-lightning")
EXP_NAME  = "tum2twin_uas_campus"
MINICONDA = Path("/content/miniconda3")
CONDA_ENV = "gspl"

# --- knobs -----------------------------------------------------------------
UNDISTORT_MAX_IMAGE_SIZE = -1   # -1 = keep native 4K. e.g. 2048 to shrink.

DOWN_SAMPLE_FACTOR = 2       # training resolution (creates images_2); no SfM re-run
MAX_STEPS          = 30000
SH_DEGREE          = 3        # kept consistent between training and the exported cfg_args

SYNC_SCENE_TO_DRIVE = True   # one-shot copy local scene -> Drive (for downstream)

# --- this repo -------------------------------------------------------------
REPO_URL = "https://github.com/satyaba/worldsplat-prune.git"
REPO_DIR = Path("/content/worldsplat-prune")
BRANCH   = "main"

for p in (RAW_DIR, DATASET_ROOT, SCENE_DIR, MODEL_DIR, SCRATCH, SCENE_LOCAL):
    p.mkdir(parents=True, exist_ok=True)

os.environ["HOME_DIR"]     = HOME_DIR
os.environ["RAW_DIR"]      = str(RAW_DIR)
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)
print("drive zips :", DRIVE_ZIP_DIR / IMAGES_ZIP_NAME)
print("             ", DRIVE_ZIP_DIR / OPF_ZIP_NAME)
print("local scene:", SCENE_LOCAL)
print("drive scene:", SCENE_DIR)
print("model      :", MODEL_DIR)

## 2b. Restore a prepared scene (skip sections 3–5)

After a runtime restart `/content` is empty, but the prepared scene may already be
on Drive (synced earlier). If so, this restores it locally and the next three
sections **skip themselves**. Set `SYNC_SCENE_TO_DRIVE=True` (config) and the
§9 sync cell keeps the Drive copy fresh.

In [ ]:
import shutil

# NEED_PREP=False when the prepared scene (images + sparse/0) is already on Drive.
NEED_PREP = not (SCENE_DIR / "sparse" / "0" / "cameras.bin").exists()
if not NEED_PREP and not (SCENE_LOCAL / "sparse" / "0" / "cameras.bin").exists():
    print("prepared scene on Drive -> restoring to local ...")
    shutil.copytree(SCENE_DIR, SCENE_LOCAL, dirs_exist_ok=True)
print("NEED_PREP =", NEED_PREP, "(False -> sections 3-5 are skipped)")

## 3. Copy zips Drive→local, then extract locally

Copying the two zips is a **few large files** (fast on Drive); extraction is
**many files** and stays on the local disk. Re-running is safe (skips existing).

In [ ]:
if NEED_PREP:
    import shutil, subprocess, time, zipfile

    DRIVE_IMAGES_ZIP = DRIVE_ZIP_DIR / IMAGES_ZIP_NAME
    DRIVE_OPF_ZIP    = DRIVE_ZIP_DIR / OPF_ZIP_NAME

    LOCAL_IMAGES_ZIP = SCRATCH / IMAGES_ZIP_NAME
    LOCAL_OPF_ZIP    = SCRATCH / OPF_ZIP_NAME

    def _copy_to_local(src, dst):
        # Copy a (large) file from mounted Drive to the local runtime, skipping if already copied.
        src, dst = Path(src), Path(dst)
        if not src.exists():
            raise FileNotFoundError(f"zip not found on Drive: {src}")
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            print("skip (local copy present):", dst.name)
            return dst
        dst.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        shutil.copy2(src, dst)
        print(f"copied {src.name}: {dst.stat().st_size / 1e9:.2f} GB in {time.time() - t0:.0f}s")
        return dst

    def _extract(zip_path, dst):
        # Unzip into a LOCAL directory, skipping if already populated.
        dst = Path(dst)
        if dst.exists() and any(dst.iterdir()):
            print("skip (extracted):", dst)
            return dst
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dst)
        print("extracted ->", dst)
        return dst

    print("Drive zip dir listing:", sorted(p.name for p in DRIVE_ZIP_DIR.glob('*'))[:20])

    _copy_to_local(DRIVE_IMAGES_ZIP, LOCAL_IMAGES_ZIP)
    _copy_to_local(DRIVE_OPF_ZIP,    LOCAL_OPF_ZIP)

    IMAGES_EXTRACT = _extract(LOCAL_IMAGES_ZIP, SCRATCH / "images_zip")
    OPF_EXTRACT    = _extract(LOCAL_OPF_ZIP,    SCRATCH / "opf_zip")

    # Locate the OPF project file.
    opf_projects = sorted(Path(OPF_EXTRACT).rglob("*.opf"))
    print("opf projects:", opf_projects)
    OPF_PROJECT = opf_projects[0] if opf_projects else None

    # Index every extracted image by basename (OPF references them by name).
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    ALL_IMAGES = [p for p in Path(IMAGES_EXTRACT).rglob("*") if p.suffix.lower() in IMG_EXTS]
    IMAGES_BY_NAME = {}
    for _p in ALL_IMAGES:
        IMAGES_BY_NAME.setdefault(_p.name, _p)
    print(f"{len(ALL_IMAGES)} images extracted ({len(IMAGES_BY_NAME)} unique names)")
    assert OPF_PROJECT is not None, "no *.opf found -- check DRIVE_ZIP_DIR / OPF_ZIP_NAME"
    assert ALL_IMAGES, "no images found -- check DRIVE_ZIP_DIR / IMAGES_ZIP_NAME"
else:
    print("NEED_PREP=False -- skipping zip copy/extract (section 3)")

## 4. OPF → COLMAP

`opf2colmap` (from `pyopf`) writes `cameras.txt` (FULL_OPENCV),
`images.txt`, `points3D.txt`. We then assemble a flat image folder matching the
names in `images.txt` by basename, so the undistorter can find them.

In [ ]:
if NEED_PREP:
    # Only what opf2colmap needs. The full `[tools]` extras pin pyproj==3.6.0, which has
    # no wheel for Colab's Python and fails to build from source (harmless, but noisy).
    !pip install -q pyopf tqdm

    COLMAP_TXT = SCRATCH / "colmap_txt"
    COLMAP_TXT.mkdir(parents=True, exist_ok=True)
    !opf2colmap "{OPF_PROJECT}" --out-dir "{COLMAP_TXT}"

    print(sorted(p.name for p in COLMAP_TXT.iterdir()))
else:
    print("NEED_PREP=False -- skipping OPF->COLMAP (section 4)")

In [ ]:
if NEED_PREP:
    import os

    # images.txt alternates pose-line / points-line; the pose line has the filename
    # in column 10. Build a flat RAW_IMAGES dir with exactly those names.
    RAW_IMAGES = SCRATCH / "raw_images"
    RAW_IMAGES.mkdir(parents=True, exist_ok=True)

    images_txt = COLMAP_TXT / "images.txt"
    data_lines = [l for l in images_txt.read_text().splitlines()
                  if l.strip() and not l.startswith("#")]
    name_lines = data_lines[0::2]
    names = [l.split()[9] for l in name_lines if len(l.split()) >= 10]
    print(f"{len(names)} images referenced by the COLMAP model")

    missing = []
    for nm in names:
        src = IMAGES_BY_NAME.get(os.path.basename(nm))
        dst = RAW_IMAGES / nm
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src is None:
            missing.append(nm)
        elif not dst.exists():
            os.symlink(src, dst)

    print(f"{len(names) - len(missing)} linked, {len(missing)} unmatched")
    if missing[:5]:
        print("  e.g.:", missing[:5])
    assert not missing, "some model images had no matching file in Images.zip"
else:
    print("NEED_PREP=False -- skipping image assembly (section 4)")

## 5. COLMAP: txt→bin + undistort → 3DGS-ready scene (local)

`colmap image_undistorter` removes FULL_OPENCV distortion and emits a PINHOLE
model. We normalise the output to `SCENE_LOCAL/images` + `SCENE_LOCAL/sparse/0`
(replicating what the repo's `convert.py` does). All of it stays **local**.

In [ ]:
if NEED_PREP:
    !apt-get -qq update >/dev/null 2>&1
    !apt-get -qq install -y colmap >/dev/null 2>&1
    !colmap -h 2>&1 | head -n 2
else:
    print("NEED_PREP=False -- skipping COLMAP install (section 5)")

In [ ]:
if NEED_PREP:
    import os

    SPARSE_BIN = SCRATCH / "sparse_bin"
    UNDIST     = SCRATCH / "undistorted"
    SPARSE_BIN.mkdir(parents=True, exist_ok=True)   # model_converter does NOT create it
    UNDIST.mkdir(parents=True, exist_ok=True)

    !colmap model_converter --input_path "{COLMAP_TXT}" --output_path "{SPARSE_BIN}" --output_type BIN
    print("bin model:", sorted(p.name for p in SPARSE_BIN.iterdir()))
    assert (SPARSE_BIN / "cameras.bin").exists(), \
        "model_converter produced no cameras.bin -- check COLMAP_TXT contents"

    !colmap image_undistorter --image_path "{RAW_IMAGES}" --input_path "{SPARSE_BIN}" --output_path "{UNDIST}" --output_type COLMAP --max_image_size {UNDISTORT_MAX_IMAGE_SIZE}
    print("undistorted tree:", sorted(p.name for p in UNDIST.iterdir()))
else:
    print("NEED_PREP=False -- skipping txt->bin + undistort (section 5)")

In [ ]:
if NEED_PREP:
    import shutil

    # images -> SCENE_LOCAL/images
    dst_img = SCENE_LOCAL / "images"
    src_img = UNDIST / "images"
    if dst_img.exists() and any(dst_img.iterdir()):
        print("skip (have images):", dst_img)
    else:
        if dst_img.exists():
            shutil.rmtree(dst_img)
        shutil.move(str(src_img), str(dst_img))   # same local disk: move is cheap

    # sparse model -> SCENE_LOCAL/sparse/0  (image_undistorter may write sparse/<files>)
    dst_sparse = SCENE_LOCAL / "sparse" / "0"
    dst_sparse.mkdir(parents=True, exist_ok=True)
    if (dst_sparse / "cameras.bin").exists():
        print("skip (have model):", dst_sparse)
    else:
        src_cams = list(UNDIST.rglob("cameras.bin"))
        if src_cams:
            src_dir = src_cams[0].parent
            for f in src_dir.iterdir():
                shutil.move(str(f), str(dst_sparse / f.name))

    n = len([p for p in dst_img.glob("*") if p.is_file()])
    ok = all((dst_sparse / f).exists() for f in ("cameras.bin", "images.bin", "points3D.bin"))
    print(f"local scene images={n} model={ok} -> {SCENE_LOCAL}")
    assert n > 0 and ok, "failed to assemble local images/ + sparse/0/"
else:
    print("NEED_PREP=False -- skipping scene assembly (section 5)")

## 6. Get this repo (evaluator + model-dir export)

Cloned/refreshed, and the two CUDA submodules are built here so the **sanity
eval** (step 10) can run in Colab's base Python. 3DGS-Lightning remains
separate (step 7).

In [ ]:
if not (REPO_DIR / ".git").exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout {BRANCH} -q
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)
!git -C {REPO_DIR} submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
!pip install -q plyfile opencv-python joblib
# --no-build-isolation: these CUDA extensions `import torch` at BUILD time, which
# pip's build isolation hides (that is why a plain `pip install` fails silently).
# The repo's rasterizer is the graphdeco `dr_aa` branch (needed by eval_worldsplat).
# CUDA >= 12.4 / C++20 need <cstdint> for std::uintptr_t & uintN_t; the upstream
# rasterizer omits the include, so inject it into every nvcc call.
!NVCC_PREPEND_FLAGS='-include cstdint' pip install --no-build-isolation {REPO_DIR}/submodules/simple-knn {REPO_DIR}/submodules/diff-gaussian-rasterization

import simple_knn._C, diff_gaussian_rasterization  # noqa: F401  (verify build)
print("base-env extensions OK; cwd:", os.getcwd())

## 7. Set up 3DGS-Lightning

A dedicated **Miniconda py3.9 + CUDA 11.8 + torch 2.0.1+cu118** env (the
combination the upstream repo pins). No Colab runtime restart is needed — the
env lives under `/content`.

In [ ]:
if not (GSL_DIR / ".git").exists():
    !git clone --depth 1 https://github.com/yzslab/gaussian-splatting-lightning.git {GSL_DIR}
!git -C {GSL_DIR} log --oneline -1

if not (MINICONDA / "bin" / "conda").exists():
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p {MINICONDA}
print("miniconda:", MINICONDA)

In [ ]:
import os, json as _json, subprocess

ENV_PREFIX = MINICONDA / "envs" / CONDA_ENV
ENV_PY     = ENV_PREFIX / "bin" / "python"
ENV_PIP    = ENV_PREFIX / "bin" / "pip"


def gspl_env():
    # The py3.9 env's own bin on PATH + CUDA_HOME/CUDA_PATH so CUDA extensions
    # compile against the env's nvcc (NOT Colab's system CUDA).
    e = dict(os.environ)
    e["CUDA_HOME"] = str(ENV_PREFIX)
    e["CUDA_PATH"] = str(ENV_PREFIX)
    e["PATH"] = str(ENV_PREFIX / "bin") + os.pathsep + e.get("PATH", "")
    # CUDA 11.8's nvcc supports host gcc <= 11 only (Colab's default is newer).
    e["CC"] = "gcc-11"
    e["CXX"] = "g++-11"
    e.setdefault("MAX_JOBS", "4")   # cap parallel compiles (avoid OOM)
    return e


def _run(args, cwd=None, env=None, capture=False):
    # capture=True is for install/build steps: on failure, print the tool's own
    # output (last lines) before raising, so the real error is visible.
    print("+", " ".join(map(str, args)), ("[cwd=%s]" % cwd) if cwd else "", flush=True)
    if capture:
        p = subprocess.run([str(a) for a in args],
                           cwd=str(cwd) if cwd else None, env=env, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if p.returncode != 0:
            print("\n".join((p.stdout or "").splitlines()[-80:]), flush=True)
            raise subprocess.CalledProcessError(p.returncode, p.args, p.stdout)
        return p
    subprocess.run([str(a) for a in args],
                   cwd=str(cwd) if cwd else None, env=env, check=True)


def gspl(*args, cwd=None, capture=False):
    # Run a command inside the 3DGS-Lightning conda env (no shell activation needed).
    return _run(args, cwd=cwd, env=gspl_env(), capture=capture)


def base(*args, capture=False):
    return _run(args, env=os.environ, capture=capture)


_conda = MINICONDA / "bin" / "conda"

# CUDA 11.8's nvcc only supports host gcc <= 11; Colab's default gcc is newer.
base("apt-get", "update", "-y", "-qq")
base("apt-get", "install", "-y", "-qq", "gcc-11", "g++-11")

_envs = _json.loads(subprocess.run([str(_conda), "env", "list", "--json"],
                                   capture_output=True, text=True).stdout).get("envs", [])
if not any(Path(e).name == CONDA_ENV for e in _envs):
    base(_conda, "create", "-n", CONDA_ENV, "python=3.9", "-y", "-q")

# CUDA 11.8 toolkit (nvcc + headers) inside the env. torch below is cu118, so the
# extensions MUST be compiled with nvcc 11.8 -- Colab's SYSTEM CUDA is 12/13 and
# would raise "detected CUDA version (13.0) mismatches ... PyTorch (11.8)".
base(_conda, "install", "-n", CONDA_ENV, "-y", "-c", "nvidia/label/cuda-11.8.0", "cuda-toolkit")


def _nvcc_release():
    nvcc = ENV_PREFIX / "bin" / "nvcc"
    if not nvcc.exists():
        return None
    txt = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True).stdout
    for tok in txt.replace(",", " ").split():
        parts = tok.split(".")
        if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
            return tok
    return None


if _nvcc_release() != "11.8":
    print("[cuda] env nvcc is", _nvcc_release(), "-- installing 11.8 components explicitly")
    base(_conda, "install", "-n", CONDA_ENV, "-y", "-c", "nvidia/label/cuda-11.8.0",
         "cuda-nvcc", "cuda-cudart-dev", "cuda-cccl")

print("env nvcc:", _nvcc_release())
assert _nvcc_release() == "11.8", \
    f"env nvcc={_nvcc_release()} but torch cu118 needs 11.8; extensions will not build"

# torch (cu118)
gspl(ENV_PIP, "install", "-q", "-r", GSL_DIR / "requirements" / "pyt201_cu118.txt", cwd=GSL_DIR, capture=True)

# ninja speeds up the CUDA-extension builds (torch otherwise falls back to distutils).
gspl(ENV_PIP, "install", "-q", "ninja", capture=True)

# Requirements. The two CUDA extensions build from source and `import torch` at
# build time; with pip's build isolation torch is hidden, so the build can fail.
# Retry once with --no-build-isolation (torch is already installed in the env).
try:
    gspl(ENV_PIP, "install", "-r", GSL_DIR / "requirements.txt", cwd=GSL_DIR, capture=True)
except subprocess.CalledProcessError:
    print("\n[retry] requirements install failed; retrying with --no-build-isolation ...\n", flush=True)
    gspl(ENV_PIP, "install", "--no-build-isolation", "-r", GSL_DIR / "requirements.txt", cwd=GSL_DIR, capture=True)

# torch 2.0.1 / torchvision 0.15.2 are built against NumPy 1.x. A stray NumPy 2.x
# breaks torch<->numpy interop ("Numpy is not available" / "_ARRAY_API not found").
gspl(ENV_PIP, "install", "-q", "numpy<2", capture=True)

In [ ]:
gspl(ENV_PY, "-c", "import torch, diff_gaussian_rasterization; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available()); print('rasterizer OK')")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 8. Train (on the local scene)

`down_sample_factor` reads downscaled `images_N` (created by
`utils/image_downsample.py`) from the **local** scene; the on-disk COLMAP model is
untouched. The `image_uint8` / caching flags are 3DGS-Lightning's documented
large-dataset no-OOM knobs. Reading images from local disk (not mounted Drive)
matters a lot here.

`split_mode reconstruction` trains on **all** views (we want a full-scene base
model; the train/test split for pruning is applied later via this repo's
`--eval`).

In [ ]:
if DOWN_SAMPLE_FACTOR and DOWN_SAMPLE_FACTOR > 1:
    gspl(ENV_PY, GSL_DIR / "utils" / "image_downsample.py", f"{SCENE_LOCAL}/images", "--factor", DOWN_SAMPLE_FACTOR, cwd=GSL_DIR)

In [ ]:
train_args = [str(GSL_DIR / "main.py"), "fit",
              "--data.path", str(SCENE_LOCAL),
              "--data.parser", "Colmap",
              "--data.parser.split_mode", "reconstruction"]
if DOWN_SAMPLE_FACTOR and DOWN_SAMPLE_FACTOR > 1:
    train_args += ["--data.parser.down_sample_factor", str(DOWN_SAMPLE_FACTOR)]
train_args += ["--data.image_uint8", "true",
               "--data.train_max_num_images_to_cache", "512",
               "--data.async_caching", "true",
               "-n", EXP_NAME, "--max_steps", str(MAX_STEPS)]
gspl(ENV_PY, *train_args, cwd=GSL_DIR)

## 8b. Retrain / finetune for large aerial scenes

The default LRs are tuned for object-scale scenes; on a campus-sized aerial scene
they cause blur and floaters. This retrains with the Inria large-scene LRs (see
gs-lightning's `configs/larger_dataset.yaml`) at full resolution.

`FINETUNE_FROM` chooses between:
* `None` — train **from scratch** (most reliable; recommended).
* a model dir/ckpt — **finetune** the existing model with the corrected LRs
  (cheaper; uses `--model.initialize_from`).

Judge quality from gs-lightning's own val renders (native resolution), not the
`-r 8` sanity eval.

In [ ]:
RETRAIN_NAME = EXP_NAME + "_v2"
RETRAIN_STEPS = 60000
RETRAIN_DOWN_SAMPLE = 1        # 1 = full resolution (L4 + image_uint8 is fine)

# None -> from scratch; or e.g. str(GSL_DIR / "outputs" / EXP_NAME) -> finetune
FINETUNE_FROM = None
# FINETUNE_FROM = str(GSL_DIR / "outputs" / EXP_NAME)

if RETRAIN_DOWN_SAMPLE and RETRAIN_DOWN_SAMPLE > 1:
    gspl(ENV_PY, GSL_DIR / "utils" / "image_downsample.py", f"{SCENE_LOCAL}/images",
         "--factor", RETRAIN_DOWN_SAMPLE, cwd=GSL_DIR)

train_args = [str(GSL_DIR / "main.py"), "fit",
              "--data.path", str(SCENE_LOCAL),
              "--data.parser", "Colmap",
              "--data.parser.split_mode", "reconstruction",
              "--data.image_uint8", "true",
              "--data.train_max_num_images_to_cache", "512",
              "--data.async_caching", "true",
              "--model.gaussian.sh_degree", str(SH_DEGREE),
              "--model.gaussian.optimization.means_lr_init", "0.000016",
              "--model.gaussian.optimization.scales_lr", "0.001",
              "--max_steps", str(RETRAIN_STEPS),
              "-n", RETRAIN_NAME]
if RETRAIN_DOWN_SAMPLE and RETRAIN_DOWN_SAMPLE > 1:
    train_args += ["--data.parser.down_sample_factor", str(RETRAIN_DOWN_SAMPLE)]
if FINETUNE_FROM:
    train_args += ["--model.initialize_from", FINETUNE_FROM]
gspl(ENV_PY, *train_args, cwd=GSL_DIR)

# the export/eval cells below follow this run
EXP_NAME = RETRAIN_NAME
MAX_STEPS = RETRAIN_STEPS   # keep §10's --iteration in sync

In [ ]:
gspl(ENV_PY, GSL_DIR / "main.py", "validate",
     "--config", str(GSL_DIR / "outputs" / EXP_NAME / "config.yaml"),
     "--save_val", cwd=GSL_DIR)
print("native-res val renders ->", GSL_DIR / "outputs" / EXP_NAME / "val")

## 9. Export → Inria-compatible model dir

`utils/ckpt2ply.py` converts the Lightning checkpoint and writes
`outputs/<exp>/point_cloud/iteration_<N>/point_cloud.ply` — the Inria path (few
files, so it can live on Drive).

We then build the rest of the model dir **deterministically from the local
COLMAP scene**: `cameras.json` (via the repo's loader, `--eval` llffhold=8
split), `input.ply`, and a `cfg_args` the evaluator/pruner can read.

> In `cfg_args` we record the **Drive** `SCENE_DIR` as `source_path` so downstream
> notebooks resolve it there; this session's sanity eval reads the faster local
> scene via `-s`. `SYNC_SCENE_TO_DRIVE` copies the finished scene to Drive once.
>
> The Inria `Camera` loader preloads all views onto the GPU, so **evaluate at a
> reduced `-r`** (step 10).

In [ ]:
gspl(ENV_PY, GSL_DIR / "utils" / "ckpt2ply.py", f"outputs/{EXP_NAME}", cwd=GSL_DIR)

In [ ]:
import sys, json, shutil, argparse
sys.path.insert(0, str(REPO_DIR))

from scene.dataset_readers import sceneLoadTypeCallbacks
from utils.camera_utils import camera_to_JSON

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1) point_cloud/ from the Lightning export
src_pc = GSL_DIR / "outputs" / EXP_NAME / "point_cloud"
assert src_pc.exists(), f"no {src_pc} -- did ckpt2ply run?"
dst_pc = MODEL_DIR / "point_cloud"
if dst_pc.exists():
    shutil.rmtree(dst_pc)
shutil.copytree(src_pc, dst_pc)
iters = sorted(int(p.name.split("_")[-1]) for p in dst_pc.glob("iteration_*"))
print("exported iterations:", iters)

# 2) cameras.json + input.ply from the LOCAL COLMAP scene (eval=True -> llffhold=8)
scene_info = sceneLoadTypeCallbacks["Colmap"](str(SCENE_LOCAL), "images", "", True, False)
cams = list(scene_info.test_cameras) + list(scene_info.train_cameras)
with open(MODEL_DIR / "cameras.json", "w") as f:
    json.dump([camera_to_JSON(i, c) for i, c in enumerate(cams)], f)
with open(scene_info.ply_path, "rb") as s, open(MODEL_DIR / "input.ply", "wb") as d:
    d.write(s.read())

# 3) cfg_args (an eval-able Namespace) -- source_path points at the DRIVE scene
cfg = argparse.Namespace(
    sh_degree=SH_DEGREE, source_path=str(SCENE_DIR), model_path=str(MODEL_DIR),
    images="images", depths="", resolution=-1, white_background=False,
    train_test_exp=False, data_device="cuda", eval=True)
with open(MODEL_DIR / "cfg_args", "w") as f:
    f.write(repr(cfg))

print("model dir:", MODEL_DIR)
print("  ", sorted(p.name for p in MODEL_DIR.iterdir()))

### Sync the prepared scene to Drive (single one-shot copy)

Written last so the many-file image set is copied once, not repeatedly. Skip if
you intend to keep the scene only on the local disk (re-run the retrieval/prep
later). `rsync`-style incremental copy would be faster on re-runs.

In [ ]:
if SYNC_SCENE_TO_DRIVE:
    SCHED = ("images", "sparse")
    for sub in SCHED:
        src = SCENE_LOCAL / sub
        dst = SCENE_DIR / sub
        if not src.exists():
            print("skip (no local", sub, ")"); continue
        if dst.exists() and any(dst.rglob("*")):
            print("skip (have on Drive):", dst)
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        print("copying", src, "->", dst)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    print("drive scene ready:", SCENE_DIR)
else:
    print("SYNC_SCENE_TO_DRIVE=False -- scene kept local at", SCENE_LOCAL)

## 10. Sanity evaluation

Renders the held-out views with this repo's evaluator, reading the **local**
scene (`-s SCENE_LOCAL`) for speed. `-r 8` keeps the preloaded image set within
VRAM; `--skip_train` evaluates only the test split. No `semantic_class/` labels
yet, so band/interior metrics are omitted (global PSNR/SSIM/LPIPS only) — that is
expected for a fresh base model.

In [ ]:
!cd {REPO_DIR} && python eval_worldsplat.py -m "{MODEL_DIR}" -s "{SCENE_LOCAL}" --eval -r 8 --skip_train --iteration {MAX_STEPS}

## 11. Persist / restore across a runtime switch

Colab **wipes `/content`** whenever you change the runtime (e.g. to a bigger GPU),
so the local scene and the gs-lightning run would be lost. The dataset zips and
the exported Inria model already live on Drive; back up the two remaining
irreplaceable things below, then after the switch run the **restore** cell.

The conda env (`/content/miniconda3`) is not persisted — re-run §6/§7 after the
switch (~10 min).

In [ ]:
import shutil

# Run this BEFORE switching the runtime (Colab wipes /content on any runtime change).
# 1) prepared scene -> Drive (many files; skip if already synced)
if (SCENE_DIR / "sparse" / "0" / "cameras.bin").exists():
    print("scene already on Drive:", SCENE_DIR)
elif (SCENE_LOCAL / "sparse" / "0" / "cameras.bin").exists():
    print("copying scene ->", SCENE_DIR)
    shutil.copytree(SCENE_LOCAL, SCENE_DIR, dirs_exist_ok=True)
else:
    print("no local scene to back up")

# 2) gs-lightning run (checkpoints + config) -> Drive (skip if absent)
run_dir = GSL_DIR / "outputs" / EXP_NAME
if run_dir.exists():
    bak = Path(HOME_DIR) / "output" / ("gsl_backup_" + EXP_NAME)
    shutil.copytree(run_dir, bak, dirs_exist_ok=True)
    print("backed up gs-lightning run ->", bak)
else:
    print("no gs-lightning run at", run_dir)

In [ ]:
import shutil

# Run AFTER re-running §1/§2 and §7 once the new runtime is up.
# EXP_NAME must match the run you backed up (e.g. "tum2twin_uas_campus_v2").
bak = Path(HOME_DIR) / "output" / ("gsl_backup_" + EXP_NAME)

# 1) prepared scene Drive -> local (fast local disk for training)
if (SCENE_LOCAL / "sparse" / "0" / "cameras.bin").exists():
    print("scene already local:", SCENE_LOCAL)
else:
    print("copying scene Drive -> local ...")
    shutil.copytree(SCENE_DIR, SCENE_LOCAL, dirs_exist_ok=True)

# 2) restore the gs-lightning run so you can resume/finetune
if bak.exists():
    shutil.copytree(bak, GSL_DIR / "outputs" / EXP_NAME, dirs_exist_ok=True)
    print("restored:", GSL_DIR / "outputs" / EXP_NAME)
    print("resume with --ckpt_path, or retrain with --model.initialize_from")
else:
    print("no backup found at", bak)

## Next steps

The base model now lives at `MODEL_DIR` in Inria layout (its `cfg_args`
`source_path` points at the Drive scene `SCENE_DIR`). To prune it:

* **Labels first** (this scene has none yet): run the `worldsplat_prune`
  front-end / `Prepare_Datasets.ipynb` to produce `semantic_*` maps under the
  scene directory.
* **Then prune** (from the repo root): `python prune_model.py -m "{MODEL_DIR}" --label_adapter composite --keep_frac 0.5`
* **Full runs**: the `experiments/` harness (`Run_Experiments.ipynb`) with
  `DATASET_NAME="tum2twin/uas_campus"`.

Caveats to remember: downsampled images (`images_2`, …) are created in the local
scene and **not** synced (only `images/` + `sparse/` are); seed variance is
±0.1–0.2 dB PSNR; and a full-resolution evaluation of all ~1,179 views needs a
reduced `-r` (or a larger GPU) because the Inria loader preloads every image.